# Thermal Comfort Model Comparison - 5-Fold Cross Validation

This notebook compares five simple model families on the model-ready `Thermal comfort` dataset using 5-fold stratified cross validation.

Models:
- logistic regression baseline
- random forest
- extra trees
- gradient boosting (`GradientBoostingClassifier`, because XGBoost is not installed in this environment)
- neural network / MLP

The code is intentionally kept simple and linear so the workflow is easy to inspect and modify.


## 1. Imports and Paths

The notebook uses `pandas`, `matplotlib`, `scikit-learn`, and the shared paper figure style from `_project_tools/paper_style.py`. Results are written to the paper-specific `04_outputs` folder.


In [ ]:
from __future__ import annotations

import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

def find_project_root() -> Path:
    for root in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [root, root / "Prism"]:
            dataset = candidate / "02_Datasets" / "model_ready" / "thermal_comfort_model_dataset.xlsx"
            tools = candidate / "03_Code" / "_project_tools"
            if dataset.exists() and tools.exists():
                return candidate
    raise FileNotFoundError("Could not locate Prism root with 02_Datasets and 03_Code/_project_tools")


PROJECT_ROOT = find_project_root()
PROJECT_TOOLS_DIR = PROJECT_ROOT / "03_Code" / "_project_tools"
sys.path.append(str(PROJECT_TOOLS_DIR))

from paper_style import COLORS as PAPER_COLORS, apply_paper_style, save_figure, style_axes

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
apply_paper_style()

SEED = 20260507
N_SPLITS = 5

DATASET_PATH = PROJECT_ROOT / "02_Datasets" / "model_ready" / "thermal_comfort_model_dataset.xlsx"
TARGET = "Thermal satisfaction 3-class"
DROP_FROM_FEATURES = ['Thermal satisfaction', 'Thermal satisfaction 3-class'] + ["TimeVote"]
OUTPUT_DIR = PROJECT_ROOT / "03_Code" / "thermal_comfort_paper" / "04_outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ["dissatisfied", "neutral", "satisfied"]
CLASS_TO_NUMBER = {label: index for index, label in enumerate(CLASS_ORDER)}


## 2. Load Dataset

The `data` sheet is already model-ready: missing values have been imputed, trace/group identifiers have been removed, and the three-class target is available.


In [ ]:
data = pd.read_excel(DATASET_PATH, sheet_name="data")

print("Dataset:", DATASET_PATH.name)
print("Rows, columns:", data.shape)
display(data.head())


In [ ]:
print("Target distribution:")
display(data[TARGET].value_counts().reindex(CLASS_ORDER))


## 3. Select Predictors

`TimeVote` is excluded because it is a trace timestamp. The target columns are also excluded from the predictors. The derived `Vote hour` and `Vote weekday` variables remain available as interpretable time-context inputs.


In [ ]:
feature_columns = [column for column in data.columns if column not in DROP_FROM_FEATURES]

X = data[feature_columns].copy()
y = data[TARGET].astype(str)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [column for column in X.columns if column not in numeric_features]

print("Target:", TARGET)
print("Number of predictors:", len(feature_columns))
print("Numeric predictors:", numeric_features)
print("Categorical / boolean predictors:", categorical_features)


## 4. Preprocessing and Models

All models use the same preprocessing:
- numeric variables are standardized,
- categorical and boolean variables are one-hot encoded.

This keeps the comparison easy to read. Tree models do not strictly require scaling, but using one shared preprocessing block avoids hidden differences between model pipelines.


In [ ]:
def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ],
        remainder="drop",
    )


models = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=8,
        class_weight="balanced",
        n_jobs=1,
        random_state=SEED,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=8,
        class_weight="balanced",
        n_jobs=1,
        random_state=SEED,
    ),
    "Gradient boosting": GradientBoostingClassifier(
        n_estimators=160,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "ANN / MLP": MLPClassifier(
        hidden_layer_sizes=(32,),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=350,
        early_stopping=False,
        random_state=SEED,
    ),
}

list(models)


## 5. Cross-Validation Helpers

The metric `ordinal_mae` treats the three target classes as ordered categories: dissatisfied = 0, neutral = 1, satisfied = 2. Lower values are better.


In [ ]:
def ordinal_mae(y_true: pd.Series, y_pred: np.ndarray) -> float:
    true_numbers = y_true.map(CLASS_TO_NUMBER).to_numpy()
    pred_numbers = pd.Series(y_pred).map(CLASS_TO_NUMBER).to_numpy()
    return mean_absolute_error(true_numbers, pred_numbers)


def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
    }


def make_pipeline(base_model):
    return Pipeline(
        steps=[
            ("preprocess", make_preprocessor()),
            ("model", base_model),
        ]
    )


## 6. Run 5-Fold Cross Validation

Each model is fitted and evaluated on the same stratified folds. Runtime is measured as fit time plus prediction time for each fold.


In [ ]:
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_rows = []
prediction_rows = []

for model_name, base_model in models.items():
    print(f"Running {model_name}...")
    for fold, (train_index, test_index) in enumerate(splitter.split(X, y), start=1):
        model = clone(base_model)
        if hasattr(model, "random_state"):
            model.set_params(random_state=SEED + fold)

        pipeline = make_pipeline(model)
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        start = time.perf_counter()
        pipeline.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - start

        start = time.perf_counter()
        y_pred = pipeline.predict(X_test)
        predict_seconds = time.perf_counter() - start

        row = {
            "model": model_name,
            "fold": fold,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
            "total_seconds": fit_seconds + predict_seconds,
        }
        row.update(evaluate_predictions(y_test, y_pred))
        fold_rows.append(row)

        for row_index, true_label, predicted_label in zip(test_index, y_test, y_pred):
            prediction_rows.append(
                {
                    "model": model_name,
                    "fold": fold,
                    "row_index": row_index,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                }
            )

fold_results = pd.DataFrame(fold_rows)
cv_predictions = pd.DataFrame(prediction_rows)
display(fold_results)


## 7. Summarize Results

The summary reports mean and standard deviation across the five folds. Higher is better for accuracy and F1 metrics; lower is better for `ordinal_mae` and runtime.


In [ ]:
metric_columns = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "ordinal_mae", "fit_seconds", "predict_seconds", "total_seconds"]

summary = (
    fold_results
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = summary.reset_index()
summary["macro_f1_per_second"] = summary["macro_f1_mean"] / summary["total_seconds_mean"]

display(summary.sort_values("macro_f1_mean", ascending=False))


In [ ]:
folds_path = TABLE_DIR / "model_comparison_5fold_folds.csv"
summary_path = TABLE_DIR / "model_comparison_5fold_summary.csv"
predictions_path = TABLE_DIR / "model_comparison_5fold_predictions.csv"

fold_results.to_csv(folds_path, index=False)
summary.to_csv(summary_path, index=False)
cv_predictions.to_csv(predictions_path, index=False)

print("Saved:")
print(folds_path)
print(summary_path)
print(predictions_path)


## 8. Plot Performance and Efficiency

Three paper-style figures are created:
- a four-panel performance comparison, one panel per metric, with colors representing models,
- a runtime comparison where lower is better,
- an efficiency trade-off plot showing macro F1 versus runtime.

Error bars show standard deviation across the five cross-validation folds.


In [ ]:
MODEL_ORDER = ["Logistic regression", "Random Forest", "Extra Trees", "Gradient boosting", "ANN / MLP"]
MODEL_LABELS = {
    "Logistic regression": "Logistic\nregression",
    "Random Forest": "Random\nForest",
    "Extra Trees": "Extra\nTrees",
    "Gradient boosting": "Gradient\nboosting",
    "ANN / MLP": "ANN /\nMLP",
}
MODEL_COLORS = {
    "Logistic regression": PAPER_COLORS["dark_blue"],
    "Random Forest": PAPER_COLORS["best"],
    "Extra Trees": PAPER_COLORS["secondary_green"],
    "Gradient boosting": PAPER_COLORS["medium"],
    "ANN / MLP": PAPER_COLORS["purple"],
}


def model_value_label_color(model: str) -> str:
    return PAPER_COLORS["text"] if model == "Gradient boosting" else "white"


def save_current_figure(fig, name: str) -> None:
    output_path = FIGURE_DIR / name
    save_figure(fig, output_path)
    print("Saved", output_path.with_suffix(".png"))
    print("Saved", output_path.with_suffix(".pdf"))


In [ ]:
performance_metrics = [
    ("accuracy_mean", "accuracy_std", "Accuracy"),
    ("balanced_accuracy_mean", "balanced_accuracy_std", "Balanced accuracy"),
    ("macro_f1_mean", "macro_f1_std", "Macro F1"),
    ("weighted_f1_mean", "weighted_f1_std", "Weighted F1"),
]

plot_df = summary.set_index("model").reindex(MODEL_ORDER)
x = np.arange(len(MODEL_ORDER))
bar_colors = [MODEL_COLORS[model] for model in MODEL_ORDER]

fig, axes = plt.subplots(ncols=4, figsize=(13.6, 3.9), sharey=True)
for ax, (mean_col, std_col, label) in zip(axes, performance_metrics):
    values = plot_df[mean_col].astype(float)
    errors = plot_df[std_col].astype(float)
    bars = ax.bar(
        x,
        values,
        yerr=errors,
        capsize=2.5,
        color=bar_colors,
        edgecolor=PAPER_COLORS["border"],
        linewidth=0.55,
        error_kw={"elinewidth": 0.8, "capthick": 0.8, "ecolor": PAPER_COLORS["axis"]},
    )
    for bar, model, value in zip(bars, MODEL_ORDER, values):
        if value >= 0.12:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value - 0.035,
                f"{value:.2f}",
                ha="center",
                va="top",
                fontsize=8.2,
                fontweight="bold",
                color=model_value_label_color(model),
            )
        else:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + 0.015,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=8.2,
                fontweight="bold",
                color=PAPER_COLORS["text"],
            )
    ax.set_title(label, fontsize=12, fontweight="bold", pad=8)
    ax.set_ylim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[model] for model in MODEL_ORDER])
    style_axes(ax, grid_axis="y")

axes[0].set_ylabel("Score")
for ax in axes[1:]:
    ax.tick_params(axis="y", labelleft=False)

fig.suptitle("Model performance by metric (5-fold cross-validation)", fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
save_current_figure(fig, "model_comparison_performance")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.4))
runtime = plot_df["total_seconds_mean"].astype(float)
runtime_std = plot_df["total_seconds_std"].astype(float)
y = np.arange(len(MODEL_ORDER))

ax.barh(
    y,
    runtime,
    xerr=runtime_std,
    capsize=2.5,
    color=[MODEL_COLORS[model] for model in MODEL_ORDER],
    edgecolor=PAPER_COLORS["border"],
    linewidth=0.55,
    error_kw={"elinewidth": 0.8, "capthick": 0.8, "ecolor": PAPER_COLORS["axis"]},
)
ax.set_title("Model runtime per fold", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean fit + predict time (s)")
ax.set_yticks(y)
ax.set_yticklabels([MODEL_LABELS[model].replace("\n", " ") for model in MODEL_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, runtime.max() * 1.22)
style_axes(ax, grid_axis="x")

for index, value in enumerate(runtime):
    ax.text(value + runtime.max() * 0.025, index, f"{value:.2f}", va="center", fontsize=9, color=PAPER_COLORS["text"])

fig.tight_layout()
save_current_figure(fig, "model_comparison_runtime")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.6))
for model in MODEL_ORDER:
    row = plot_df.loc[model]
    ax.errorbar(
        row["total_seconds_mean"],
        row["macro_f1_mean"],
        xerr=row["total_seconds_std"],
        yerr=row["macro_f1_std"],
        fmt="o",
        markersize=7,
        color=MODEL_COLORS[model],
        ecolor=PAPER_COLORS["axis"],
        elinewidth=0.8,
        capsize=2.5,
        markeredgecolor="white",
        markeredgewidth=0.8,
        label=model,
    )
    ax.annotate(
        MODEL_LABELS[model].replace("\n", " "),
        (row["total_seconds_mean"], row["macro_f1_mean"]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=PAPER_COLORS["text"],
    )

ax.set_title("Performance-efficiency trade-off", fontsize=14, fontweight="bold")
ax.set_xlabel("Mean fit + predict time per fold (s)")
ax.set_ylabel("Mean macro F1")
ax.set_xlim(0, plot_df["total_seconds_mean"].max() * 1.28)
macro_f1_min = plot_df["macro_f1_mean"].min()
macro_f1_max = plot_df["macro_f1_mean"].max()
ax.set_ylim(max(0, macro_f1_min - 0.08), min(1, macro_f1_max + 0.08))
style_axes(ax, grid_axis="both")
fig.tight_layout()
save_current_figure(fig, "model_comparison_efficiency_tradeoff")
plt.show()


## 9. Confusion Matrices

The confusion matrices use out-of-fold predictions from the same 5-fold cross-validation. Rows are normalized to 100%, so each row shows how a true class was predicted.


In [ ]:
CLASS_DISPLAY_LABELS = ["Dissatisfied", "Neutral", "Satisfied"]
CONFUSION_CMAP = LinearSegmentedColormap.from_list(
    "paper_confusion",
    ["#FFFFFF", "#D9DEE3", PAPER_COLORS["dark_blue"]],
)

confusion_rows = []
fig, axes = plt.subplots(ncols=len(MODEL_ORDER), figsize=(13.6, 3.6), sharex=True, sharey=True)

for ax, model in zip(axes, MODEL_ORDER):
    model_predictions = cv_predictions[cv_predictions["model"] == model]
    counts = confusion_matrix(
        model_predictions["true_label"],
        model_predictions["predicted_label"],
        labels=CLASS_ORDER,
    )
    row_totals = counts.sum(axis=1, keepdims=True)
    normalized = counts / np.where(row_totals == 0, 1, row_totals) * 100

    image = ax.imshow(normalized, vmin=0, vmax=100, cmap=CONFUSION_CMAP)
    ax.set_title(MODEL_LABELS[model].replace("\n", " "), fontsize=10.5, fontweight="bold", pad=8)
    ax.set_xticks(np.arange(len(CLASS_ORDER)))
    ax.set_yticks(np.arange(len(CLASS_ORDER)))
    ax.set_xticklabels(CLASS_DISPLAY_LABELS, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_DISPLAY_LABELS)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    for true_index, true_label in enumerate(CLASS_ORDER):
        for predicted_index, predicted_label in enumerate(CLASS_ORDER):
            percent = normalized[true_index, predicted_index]
            count = counts[true_index, predicted_index]
            confusion_rows.append(
                {
                    "model": model,
                    "true_label": true_label,
                    "predicted_label": predicted_label,
                    "count": int(count),
                    "row_percent": percent,
                }
            )
            ax.text(
                predicted_index,
                true_index,
                f"{percent:.0f}%\n{count}",
                ha="center",
                va="center",
                fontsize=8.2,
                color="white" if percent >= 50 else PAPER_COLORS["text"],
            )

axes[0].set_ylabel("True class")
for ax in axes:
    ax.set_xlabel("Predicted class")

fig.suptitle("Out-of-fold confusion matrices (5-fold cross-validation)", fontsize=14, fontweight="bold", y=0.98)
fig.subplots_adjust(wspace=0.22, bottom=0.27, top=0.78, right=0.92)
colorbar_axis = fig.add_axes([0.935, 0.30, 0.012, 0.43])
colorbar = fig.colorbar(image, cax=colorbar_axis)
colorbar.set_label("Row-normalized share (%)")

confusion_table = pd.DataFrame(confusion_rows)
confusion_path = TABLE_DIR / "model_confusion_matrices.csv"
confusion_table.to_csv(confusion_path, index=False)
print("Saved", confusion_path)

save_current_figure(fig, "model_comparison_confusion_matrices")
plt.show()


## 10. Quick Interpretation Table

This final table is sorted by macro F1, which is useful for imbalanced three-class targets because each class contributes equally to the score.


In [ ]:
interpretation_columns = [
    "model",
    "accuracy_mean",
    "balanced_accuracy_mean",
    "macro_f1_mean",
    "weighted_f1_mean",
    "ordinal_mae_mean",
    "total_seconds_mean",
    "macro_f1_per_second",
]

display(summary[interpretation_columns].sort_values("macro_f1_mean", ascending=False))
